# Day 13 — Embedding Explorer: TF-IDF vs OpenAI Embeddings

### Tools
- Python
- OpenAI Embeddings API
- NumPy
- matplotlib

## Project goal

We will compare sparse TF-IDF similarity with dense semantic embeddings on the **same 20 sentences and sentence pairs**.

The project will:
1. Generate OpenAI embeddings for 20 sentences.
2. Calculate cosine similarity for sentence pairs.
3. Compare embedding scores with the Day 9 TF-IDF baseline.
4. Find five strong semantic-vs-lexical examples.
5. Build `embed_and_recommend(query_sentence, corpus_sentences)`.
6. Cluster the 20 embeddings into four groups with K-means.
7. Verify cluster/topic alignment.
8. Measure API generation time and actual token usage.
9. Estimate API cost.
10. Explain sparse TF-IDF vs dense embeddings.

> **Important:** This notebook makes a real API request. Never paste an API key directly into the notebook.

# 1. TF-IDF vs embeddings

### TF-IDF

TF-IDF is mainly a **lexical** representation. It represents text using vocabulary features, so sentences with different wording can receive low similarity even when a human sees them as paraphrases.

### Embeddings

An embedding model converts a sentence into a dense numerical vector that captures information useful for semantic relatedness.

Conceptually:

```text
sentence → embedding model → dense vector
```

OpenAI describes embeddings as numerical representations useful for relatedness, search, clustering, recommendations, anomaly detection, and classification. citeturn2search0

# 2. Install packages if necessary

Run this only if the imports below fail because a package is missing.

In [ ]:
# %pip install openai scikit-learn matplotlib numpy

### New command: `%pip install`

`%pip` is a Jupyter/IPython command used to install packages into the notebook environment.

# 3. Import libraries

In [ ]:
import os
import time
from itertools import combinations

import numpy as np
import matplotlib.pyplot as plt

from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans

### New imports

- `os` reads environment variables.
- `time` measures elapsed time.
- `combinations` generates every unique sentence pair.
- `numpy` stores vectors and performs numerical operations.
- `matplotlib` creates graphs.
- `OpenAI` is the official Python SDK client.
- `TfidfVectorizer` recreates the Day 9 TF-IDF baseline.
- `cosine_similarity` compares vectors.
- `KMeans` performs the four-cluster experiment.

# 4. Configure the OpenAI API key safely

OpenAI's quickstart recommends storing the API key as an environment variable rather than hard-coding it in source code. citeturn1search1

Example terminal commands:

```text
export OPENAI_API_KEY="your_api_key_here"
```

Windows PowerShell:

```text
$env:OPENAI_API_KEY="your_api_key_here"
```

In [ ]:
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise RuntimeError(
        "OPENAI_API_KEY was not found. "
        "Set it as an environment variable before running the API cell."
    )

client = OpenAI(api_key=api_key)

print("OpenAI client created successfully.")

### Why `os.getenv()`?

```python
os.getenv("OPENAI_API_KEY")
```

reads the secret from the operating system environment.

This is safer than putting a secret directly in a notebook that might be shared.

The `OpenAI` object is the client through which Python communicates with the API.

# 5. Create the 20-sentence dataset

We need four topics with five sentences each:

- Sports
- Technology
- Cooking
- Travel

Several pairs are intentionally written as paraphrases with different vocabulary.

In [ ]:
sentences = [
    # Sports
    "The goalkeeper blocked the striker's powerful shot.",
    "The keeper stopped the forward's fierce attempt at goal.",
    "The basketball team won the match after scoring in the final minute.",
    "The players secured victory with a basket during the last minute.",
    "Regular training improves an athlete's speed, strength, and endurance.",

    # Technology
    "Cloud computing lets companies run software on remote internet servers.",
    "Businesses can use online computing resources instead of maintaining local machines.",
    "A smartphone can store photos, apps, messages, and other personal data.",
    "Mobile devices allow users to carry applications and digital information anywhere.",
    "Strong passwords and multi-factor authentication improve account security.",

    # Cooking
    "The chef sautéed chopped onions in olive oil before adding the vegetables.",
    "The cook gently fried diced onions in oil before mixing in the vegetables.",
    "Baking bread requires flour, water, yeast, salt, and sufficient rising time.",
    "Bread dough needs time to ferment before it is placed in the oven.",
    "Fresh herbs can add aroma and flavor to a simple pasta dish.",

    # Travel
    "Travelers can explore historic streets and museums during a city holiday.",
    "Visitors may discover old neighborhoods and cultural attractions on a trip.",
    "A lightweight backpack is useful for hikers who travel through mountain trails.",
    "Hikers often prefer compact luggage when walking long distances in the mountains.",
    "Checking local transport schedules can make a journey through a new city easier.",
]

topics = (
    ["Sports"] * 5
    + ["Technology"] * 5
    + ["Cooking"] * 5
    + ["Travel"] * 5
)

print("Number of sentences:", len(sentences))
print("Number of topic labels:", len(topics))

### Why keep `topics` separately?

K-means will not receive our topic labels. It only sees the embedding vectors.

We keep the original labels so we can evaluate whether the clusters discovered from the vectors correspond to the four known topics.

In [ ]:
for index, (sentence, topic) in enumerate(zip(sentences, topics), start=1):
    print(f"{index:02d}. [{topic}] {sentence}")

### New command: `zip()`

`zip(sentences, topics)` pairs the sentence at position `i` with the topic at position `i`.

`enumerate(..., start=1)` gives us a counter beginning at 1 instead of 0.

# 6. Build the Day 9 TF-IDF baseline

We calculate TF-IDF locally before using the API. This gives us a direct baseline for the same sentences.

In [ ]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(sentences)

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(tfidf_vectorizer.vocabulary_))

### `fit_transform()`

`fit_transform()` does two things:

1. **fit** — learn vocabulary and IDF statistics.
2. **transform** — convert the sentences into TF-IDF vectors.

The resulting matrix is sparse because most sentences use only a small part of the vocabulary.

# 7. Define five candidate paraphrase pairs

We will calculate the actual scores instead of inventing expected numbers.

In [ ]:
paraphrase_pairs = [
    (0, 1),   # goalkeeper / keeper
    (2, 3),   # basketball victory / players secured victory
    (5, 6),   # cloud computing / online computing resources
    (10, 11), # sauté onions / fry diced onions
    (15, 16), # historic streets / old neighborhoods
]

for left, right in paraphrase_pairs:
    print(f"{left + 1} ↔ {right + 1}")
    print("A:", sentences[left])
    print("B:", sentences[right])
    print()

# 8. Calculate TF-IDF similarity for all 190 unique pairs

For 20 sentences:

```text
20 × 19 / 2 = 190
```

unique unordered pairs exist.

In [ ]:
pair_indices = list(combinations(range(len(sentences)), 2))

tfidf_pair_scores = {}

for i, j in pair_indices:
    score = cosine_similarity(
        tfidf_matrix[i],
        tfidf_matrix[j]
    )[0, 0]

    tfidf_pair_scores[(i, j)] = float(score)

print("Unique sentence pairs:", len(pair_indices))

### Why `combinations()`?

Comparing sentence 1 with sentence 2 gives the same cosine similarity as sentence 2 with sentence 1.

`combinations(..., 2)` therefore avoids duplicate work.

# 9. Generate OpenAI embeddings

We use `text-embedding-3-small`.

OpenAI currently lists this model at **$0.02 per 1 million input tokens** and describes embeddings as useful for relatedness, search, clustering, recommendations, anomaly detection, and classification. citeturn2search0

The embeddings endpoint is `/v1/embeddings`, and the official Python SDK exposes it through the `OpenAI` client. citeturn1search0turn1search1

This cell makes a real API request and can incur a small usage charge.

In [ ]:
EMBEDDING_MODEL = "text-embedding-3-small"

start_time = time.perf_counter()

response = client.embeddings.create(
    model=EMBEDDING_MODEL,
    input=sentences
)

elapsed_seconds = time.perf_counter() - start_time

embeddings = np.array(
    [item.embedding for item in response.data],
    dtype=np.float32
)

print("Embedding generation complete.")
print("Elapsed time:", round(elapsed_seconds, 3), "seconds")
print("Embedding matrix shape:", embeddings.shape)

### New API command

```python
client.embeddings.create(
    model=EMBEDDING_MODEL,
    input=sentences
)
```

means:

- `client` — our API connection
- `embeddings` — use the embedding service
- `create()` — request embeddings
- `model=` — choose the model
- `input=` — provide the text to embed

Because `input` is a list, all 20 sentences are sent in one request.

# 10. Inspect the dense vectors

In [ ]:
print("Number of embeddings:", len(embeddings))
print("Dimensions per embedding:", embeddings.shape[1])
print()
print("First 10 values of the first embedding:")
print(embeddings[0][:10])

### Sparse vs dense

TF-IDF is typically sparse:

```text
[0, 0, 0.42, 0, 0, 0.17, ...]
```

Embeddings are dense:

```text
[0.012, -0.081, 0.143, 0.027, ...]
```

The embedding dimensions are not interpreted as individual words. Meaning is distributed across the vector.

# 11. Measure API time and estimated cost

Embedding responses provide usage information. We use the returned input-token count instead of guessing the number of tokens.

In [ ]:
usage = getattr(response, "usage", None)

if usage is not None:
    input_tokens = getattr(usage, "prompt_tokens", None)
else:
    input_tokens = None

PRICE_PER_MILLION_TOKENS = 0.02

print("Embedding generation time:", round(elapsed_seconds, 4), "seconds")

if input_tokens is not None:
    estimated_cost = (
        input_tokens / 1_000_000
    ) * PRICE_PER_MILLION_TOKENS

    print("Embedding input tokens:", input_tokens)
    print(f"Estimated API cost: ${estimated_cost:.8f}")
else:
    print("Token usage was not returned; exact cost cannot be calculated from this response.")

### Cost formula

```text
cost =
(input tokens / 1,000,000)
× $0.02
```

OpenAI's current model page lists `text-embedding-3-small` at $0.02 per 1M input tokens. citeturn2search0

Using actual usage is better than estimating from characters because tokens are model-specific pieces of text. citeturn0search3

# 12. Calculate embedding cosine similarity for all 190 pairs

In [ ]:
embedding_pair_scores = {}

for i, j in pair_indices:
    score = cosine_similarity(
        embeddings[i].reshape(1, -1),
        embeddings[j].reshape(1, -1)
    )[0, 0]

    embedding_pair_scores[(i, j)] = float(score)

print("Embedding pair scores calculated:", len(embedding_pair_scores))

### Why `reshape(1, -1)`?

One embedding is a one-dimensional array.

Scikit-learn expects a two-dimensional matrix shaped like:

```text
number_of_samples × number_of_features
```

So `reshape(1, -1)` means:

```text
1 sample × all embedding dimensions
```

The `-1` lets NumPy calculate the required number of columns automatically.

# 13. Compare the five candidate paraphrase pairs

In [ ]:
for i, j in paraphrase_pairs:
    tfidf_score = tfidf_pair_scores[(i, j)]
    embedding_score = embedding_pair_scores[(i, j)]

    print("=" * 100)
    print(f"Pair: {i + 1} ↔ {j + 1}")
    print("Sentence A:", sentences[i])
    print("Sentence B:", sentences[j])
    print(f"TF-IDF similarity:     {tfidf_score:.4f}")
    print(f"Embedding similarity:  {embedding_score:.4f}")
    print(f"Embedding - TF-IDF:    {embedding_score - tfidf_score:+.4f}")

# 14. Find five low-TF-IDF / high-embedding examples

The exact scores come from the live embedding API, so we should not hard-code fake results.

First we look for pairs satisfying:

```text
TF-IDF < 0.30
Embedding > 0.70
```

If fewer than five pairs satisfy both conditions, we report the five largest embedding-over-TF-IDF gaps instead.

In [ ]:
LOW_TFIDF = 0.30
HIGH_EMBEDDING = 0.70

qualified_pairs = [
    (pair, tfidf_pair_scores[pair], embedding_pair_scores[pair])
    for pair in pair_indices
    if tfidf_pair_scores[pair] < LOW_TFIDF
    and embedding_pair_scores[pair] > HIGH_EMBEDDING
]

qualified_pairs.sort(
    key=lambda item: item[2] - item[1],
    reverse=True
)

print(
    f"Pairs meeting TF-IDF < {LOW_TFIDF} "
    f"and embedding > {HIGH_EMBEDDING}: "
    f"{len(qualified_pairs)}"
)

for rank, (pair, tfidf_score, embedding_score) in enumerate(
    qualified_pairs[:5], start=1
):
    i, j = pair
    print("=" * 100)
    print(f"{rank}. Pair {i + 1} ↔ {j + 1}")
    print("A:", sentences[i])
    print("B:", sentences[j])
    print(f"TF-IDF: {tfidf_score:.4f}")
    print(f"Embedding: {embedding_score:.4f}")

If the live model produces fewer than five pairs above both thresholds, that is not a reason to fabricate values. The next cell provides the five strongest semantic-over-lexical gaps so the experiment remains honest and reproducible.

In [ ]:
if len(qualified_pairs) < 5:
    fallback_pairs = sorted(
        pair_indices,
        key=lambda pair: (
            embedding_pair_scores[pair]
            - tfidf_pair_scores[pair]
        ),
        reverse=True
    )[:5]

    print("Fewer than five pairs met both thresholds.")
    print("Showing the five largest embedding-over-TF-IDF gaps instead.")

    for rank, pair in enumerate(fallback_pairs, start=1):
        i, j = pair
        print("=" * 100)
        print(f"{rank}. Pair {i + 1} ↔ {j + 1}")
        print("A:", sentences[i])
        print("B:", sentences[j])
        print(
            f"TF-IDF: {tfidf_pair_scores[pair]:.4f} | "
            f"Embedding: {embedding_pair_scores[pair]:.4f} | "
            f"Gap: {embedding_pair_scores[pair] - tfidf_pair_scores[pair]:+.4f}"
        )

# 15. Build `embed_and_recommend(query_sentence, corpus_sentences)`

The function will:

1. validate the inputs
2. embed the query
3. reuse the existing corpus embeddings when possible
4. calculate cosine similarity
5. sort the results
6. return the top three sentences with scores

In a real application, corpus embeddings should be stored and reused rather than regenerated for every query.

In [ ]:
def embed_and_recommend(query_sentence, corpus_sentences):
    """Return the three most semantically similar corpus sentences."""

    if not isinstance(query_sentence, str) or not query_sentence.strip():
        raise ValueError("query_sentence must be a non-empty string.")

    if not isinstance(corpus_sentences, list) or not corpus_sentences:
        raise ValueError("corpus_sentences must be a non-empty list.")

    query_response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=[query_sentence]
    )

    query_embedding = np.array(
        query_response.data[0].embedding,
        dtype=np.float32
    ).reshape(1, -1)

    if corpus_sentences == sentences:
        corpus_embeddings = embeddings
    else:
        corpus_response = client.embeddings.create(
            model=EMBEDDING_MODEL,
            input=corpus_sentences
        )

        corpus_embeddings = np.array(
            [item.embedding for item in corpus_response.data],
            dtype=np.float32
        )

    scores = cosine_similarity(
        query_embedding,
        corpus_embeddings
    )[0]

    ranked_indices = np.argsort(scores)[::-1][:3]

    return [
        {
            "rank": rank,
            "sentence": corpus_sentences[index],
            "similarity": float(scores[index])
        }
        for rank, index in enumerate(ranked_indices, start=1)
    ]

### Important new lines

`raise ValueError(...)` stops the function with a useful input error.

`np.argsort(scores)[::-1]` sorts document indices from highest score to lowest score.

The function returns dictionaries so that the result is easy to read and extend later.

# 16. Test `embed_and_recommend()`

In [ ]:
query = "What helps an athlete become faster and stronger?"

recommendations = embed_and_recommend(
    query,
    sentences
)

print("Query:", query)
print()

for result in recommendations:
    print(
        f"{result['rank']}. "
        f"score={result['similarity']:.4f} | "
        f"{result['sentence']}"
    )

# 17. K-means clustering

Now we ask whether the embedding vectors naturally separate into our four topics.

K-means receives only the 20 vectors. It does **not** receive:

```text
Sports
Technology
Cooking
Travel
```

as labels.

In [ ]:
kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(embeddings)

print("Cluster labels:")
print(cluster_labels)

### New command: `KMeans`

`n_clusters=4` asks for four groups.

`random_state=42` makes initialization reproducible.

`n_init=10` runs multiple initializations and keeps a strong result.

Cluster numbers are arbitrary. Cluster `0` does not inherently mean Sports; we must inspect the contents.

# 18. Inspect the clusters

In [ ]:
for cluster_id in range(4):
    print("=" * 100)
    print("CLUSTER", cluster_id)

    indices = np.where(cluster_labels == cluster_id)[0]

    for index in indices:
        print(f"[{topics[index]}] {sentences[index]}")

    print()

A strong result should show that each cluster contains mostly one original topic.

This is an experiment showing whether semantic geometry aligns with our manually assigned topics.

# 19. Calculate cluster purity

For each cluster, we count which original topic is most common.

The overall weighted purity is:

```text
correctly represented dominant-topic items
------------------------------------------
total items
```

In [ ]:
def cluster_purity(labels, true_topics):
    total_correct = 0

    for cluster_id in sorted(set(labels)):
        indices = np.where(labels == cluster_id)[0]

        cluster_topics = [
            true_topics[index]
            for index in indices
        ]

        counts = {
            topic: cluster_topics.count(topic)
            for topic in set(cluster_topics)
        }

        total_correct += max(counts.values())

    return total_correct / len(true_topics)


purity = cluster_purity(cluster_labels, topics)

print(f"Overall cluster purity: {purity:.3f}")

A purity of `1.000` means every cluster can be assigned one topic without any mixed-topic errors.

Lower values mean the clusters contain more topic mixing.

# 20. Visualise the embedding clusters

Embedding vectors have many dimensions, so we cannot directly draw them in two dimensions.

We use NumPy SVD to make a two-dimensional visualization.

The clustering itself still uses the **full embedding vectors**.

In [ ]:
centered_embeddings = embeddings - embeddings.mean(axis=0)

U, S, Vt = np.linalg.svd(
    centered_embeddings,
    full_matrices=False
)

points_2d = U[:, :2] * S[:2]

plt.figure(figsize=(10, 7))

for topic in sorted(set(topics)):
    indices = [
        index
        for index, item_topic in enumerate(topics)
        if item_topic == topic
    ]

    plt.scatter(
        points_2d[indices, 0],
        points_2d[indices, 1],
        label=topic
    )

    for index in indices:
        plt.annotate(
            str(index + 1),
            (
                points_2d[index, 0],
                points_2d[index, 1]
            )
        )

plt.xlabel("Projection dimension 1")
plt.ylabel("Projection dimension 2")
plt.title("20 Sentence Embeddings — Topic Visualization")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()

### New NumPy command: `np.linalg.svd()`

SVD decomposes a matrix into useful mathematical components.

We keep the first two directions only for visualization.

This does **not** change the vectors used by K-means.

# 21. Compare TF-IDF and embedding similarity for all 190 pairs

Each point represents one sentence pair.

- x-axis = TF-IDF similarity
- y-axis = embedding similarity

Points far above the diagonal-like relationship show cases where embeddings find more semantic relatedness than word overlap alone.

In [ ]:
tfidf_values = np.array([
    tfidf_pair_scores[pair]
    for pair in pair_indices
])

embedding_values = np.array([
    embedding_pair_scores[pair]
    for pair in pair_indices
])

plt.figure(figsize=(9, 7))

plt.scatter(
    tfidf_values,
    embedding_values,
    alpha=0.7
)

plt.xlabel("TF-IDF cosine similarity")
plt.ylabel("Embedding cosine similarity")
plt.title("TF-IDF vs Embedding Similarity — 190 Sentence Pairs")
plt.grid(True, alpha=0.25)
plt.show()

# 22. Print the five largest semantic-over-lexical gaps

In [ ]:
largest_gaps = sorted(
    pair_indices,
    key=lambda pair: (
        embedding_pair_scores[pair]
        - tfidf_pair_scores[pair]
    ),
    reverse=True
)[:5]

for rank, pair in enumerate(largest_gaps, start=1):
    i, j = pair

    print("=" * 100)
    print(f"#{rank} — sentences {i + 1} and {j + 1}")
    print("A:", sentences[i])
    print("B:", sentences[j])
    print(f"TF-IDF similarity:    {tfidf_pair_scores[pair]:.4f}")
    print(f"Embedding similarity: {embedding_pair_scores[pair]:.4f}")
    print(
        f"Semantic gain:        "
        f"{embedding_pair_scores[pair] - tfidf_pair_scores[pair]:+.4f}"
    )

# 23. Written explanation: sparse TF-IDF vs dense embeddings

## Sparse TF-IDF

TF-IDF is a sparse lexical representation.

Its dimensions correspond to vocabulary features, and the vector reflects which words appear and how informative they are.

**Advantages**
- fast
- local
- inexpensive
- interpretable
- excellent for exact keyword matching

**Limitation**

A human may see:

```text
"the keeper stopped the attempt"
```

and:

```text
"the goalkeeper blocked the shot"
```

as highly related, while TF-IDF may see limited shared vocabulary.

## Dense embeddings

Embeddings are dense numerical representations produced by a trained embedding model.

They encode semantic relationships in vector space, so sentences with related meanings can be close even when they use different vocabulary.

This makes embeddings useful for semantic search, recommendations, clustering, and relatedness tasks. citeturn2search0

## Why embeddings generalise across vocabulary

The model has learned statistical relationships among language patterns. Meaning is distributed across the vector rather than being represented by one independent dimension per word.

Therefore:

```text
"keeper"
"goalkeeper"
```

can contribute to similar semantic representations even when the exact strings differ.

## Important limitation

Embedding similarity is not a guarantee of logical equivalence or factual correctness. It can still struggle with negation, numbers, specialized terminology, and subtle context.

So embeddings should be treated as a powerful semantic representation, not a perfect understanding of language.

# 24. Submission checklist

- [x] 20 semantically diverse sentences.
- [x] Four topics: sports, technology, cooking, travel.
- [x] OpenAI embeddings generated.
- [x] TF-IDF baseline calculated on the same sentences.
- [x] Cosine similarity calculated for the same sentence pairs.
- [x] Five candidate paraphrase pairs included.
- [x] Automatic search for five low-TF-IDF/high-embedding examples.
- [x] `embed_and_recommend(query_sentence, corpus_sentences)` implemented.
- [x] Top three semantic recommendations returned.
- [x] K-means with four clusters implemented.
- [x] Cluster/topic alignment inspected.
- [x] Cluster purity calculated.
- [x] Embedding generation time measured.
- [x] Actual API token usage read when available.
- [x] Estimated API cost calculated.
- [x] TF-IDF vs dense embedding explanation included.
- [x] Matplotlib visualisations included.

## Final takeaway

```text
Day 9
TF-IDF
  ↓
lexical similarity

Day 10–11
TF-IDF retrieval
  ↓
document ranking

Day 12
Chunking
  ↓
better retrieval units

Day 13
Embeddings
  ↓
semantic similarity
  ↓
paraphrase-aware retrieval
  ↓
semantic clustering
```

The key lesson is:

> **TF-IDF mainly asks whether texts use similar words, while embeddings are designed to capture broader semantic relatedness.**